# Lottery Number Checker

steps

   1. Check for latest image in image folder
      - Import packages and read files
      - Organize folders and find the latest images
      - Check on tesseract installation for OCR operations
   2. Read image text w/Manual Steps
      - Crop Image w/MS Paint
        - process will open MS Pain, crop then save/close
      - Apply Green Dots on outer corners of #s
        - must be pure green RGB (0,255,0) and use 5 pixel width and the brush tool
        - process will open MS Paint, crop then save/close
      - Image will be processed for maximal readibility (color + perspective correction)
   3. Connect to Lotto Max Website and check the latest draw
   4. Bonus: Max-Millions & Max-Plus
      - If max millions (jackpot > $50MM) - check (no API)
      - Copy and paste the max million #s from:
      https://www.olg.ca/en/lottery/play-lotto-max-encore/hub.html
         - Run the checker, export maxmillion_results.csv
      - Copy and paste the max plus #s from:
      https://www.olg.ca/en/lottery/play-lotto-max-encore/hub.html
   

## 1. Check for latest images

##### import packages and read files

In [ ]:
# import packages

import os
import cv2
import numpy as np
import pytesseract
import numpy as np
import pandas as pd
import PIL 
import requests

##### Organize folders and find the latest images

In [ ]:
# Get the list of files in the directory
files = os.listdir('images')

# Sort the list of files by the last modified date in descending order
files.sort(key=lambda x: os.path.getmtime('images/' + x), reverse=True)

# Print the sorted list of files
for i,x in enumerate(files):
    print(i+1,x)

fname = os.path.join(os.getcwd(),'images', files[int(input("Enter the file number to check: ")) - 1])

### Check on tesseract installation for OCR operations

In [ ]:
# Define the folder path where Tesseract is installed
folder_path = f"C:\\Users\\{os.getlogin()}\\Programs\\Tesseract-OCR"

# Define the name of the Tesseract executable file
tesseract_exe = "tesseract.exe"

# Check if Tesseract is installed in the specified folder path
if os.path.exists(os.path.join(folder_path, tesseract_exe)):
    # Assign the path to the Tesseract executable file to pytesseract.tesseract_cmd
    pytesseract.pytesseract.tesseract_cmd = os.path.join(folder_path, tesseract_exe)
    print("Tesseract is installed and located in the specified folder path.")

else:
    print("Tesseract is not installed or not located in the specified folder path.")

## 2. Read image text w/Manual Steps

In [ ]:

# 1. Load original image
image = PIL.Image.open(fname)

# 2. Save a working copy for manual cropping/straightening
manual_image_path = os.path.join(
    os.getcwd(), 'images', 'manual_crop.png'
)
os.makedirs(os.path.dirname(manual_image_path), exist_ok=True)
image.save(manual_image_path)

# 3. Open original photo in MS Paint
os.system(f'mspaint "{manual_image_path}"')




### Crop Image w/MS Paint
        - process will open MS Pain, crop then save/close

In [ ]:

# 4. Reload your manually cropped/straightened image
image = PIL.Image.open(manual_image_path).convert('RGB')


# 5. Grayscale
image = image.convert('L')

# 6. Upscale
import numpy as np

gray = np.array(image)
gray = cv2.resize(
    gray, None, fx=2, fy=2,
    interpolation=cv2.INTER_CUBIC
)

# 7. Adaptive thresholding
processed = cv2.adaptiveThreshold(
    gray, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    31, 8
)

# 8. Save preprocessed image
preprocessed_image_path = os.path.join(
    os.getcwd(), 'images', 'preprocessed_image.png'
)
cv2.imwrite(preprocessed_image_path, processed)

### Apply Green Dots on outer corners of #s
        - must be pure green RGB (0,255,0) and use 5 pixel width and the brush tool
        - process will open MS Paint, crop then save/close

In [ ]:
# 9. Open preprocessed image in Paint to add four green corner markers
# (Make sure preprocessed_image_path points to your cropped image without dots yet)
os.system(f'mspaint "{preprocessed_image_path}"')
#input("Press Enter *after* you have added the 4 green dots in MS Paint and saved the file...")

# 10. Reload the marked image with your four green corner markers
marked = cv2.imread(preprocessed_image_path)
if marked is None:
    raise FileNotFoundError(preprocessed_image_path)

# To apply the perspective matrix to the clean image later, let's load a clean copy 
# (or re-load your cropped image before you added dots, assuming you saved it separately).
# If preprocessed_image_path had the dots saved over it, we should load a clean version.
# Let's assume 'clean_cropped' is your image right before step 9:
clean_image = clean_cropped.copy() if 'clean_cropped' in locals() else marked.copy()

# 11. Detect pure green markers
lower_green = np.array([0, 200, 0], dtype=np.uint8)
upper_green = np.array([100, 255, 100], dtype=np.uint8)

mask = cv2.inRange(marked, lower_green, upper_green)

# Find connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

dots = []
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if 5 <= area <= 5000:
        dots.append(centroids[i])

if len(dots) != 4:
    raise ValueError(
        f"Expected 4 green dots, found {len(dots)}. "
        "Ensure the markers are distinct and pure green."
    )

# 12. Order dots: top-left, top-right, bottom-right, bottom-left
pts = np.array(dots, dtype=np.float32)

sums = pts.sum(axis=1)
diffs = np.diff(pts, axis=1).ravel()

top_left = pts[np.argmin(sums)]
bottom_right = pts[np.argmax(sums)]
top_right = pts[np.argmin(diffs)]
bottom_left = pts[np.argmax(diffs)]

src = np.array([
    top_left,
    top_right,
    bottom_right,
    bottom_left
], dtype=np.float32)

# 13. Determine destination dimensions based on the actual distances 
# between your chosen corner dots to prevent scaling distortion
width_A = np.sqrt(
    ((bottom_right[0] - bottom_left[0]) ** 2) + ((bottom_right[1] - bottom_left[1]) ** 2)
)
width_B = np.sqrt(
    ((top_right[0] - top_left[0]) ** 2) + ((top_right[1] - top_left[1]) ** 2)
)
maxWidth = max(int(width_A), int(width_B))

height_A = np.sqrt(
    ((top_right[0] - bottom_right[0]) ** 2) + ((top_right[1] - bottom_right[1]) ** 2)
)
height_B = np.sqrt(
    ((top_left[0] - bottom_left[0]) ** 2) + ((top_left[1] - bottom_left[1]) ** 2)
)
maxHeight = max(int(height_A), int(height_B))

dst = np.array([
    [0, 0],
    [maxWidth - 1, 0],
    [maxWidth - 1, maxHeight - 1],
    [0, maxHeight - 1]
], dtype=np.float32)

# 14. Apply perspective correction
matrix = cv2.getPerspectiveTransform(src, dst)

# Warp the clean image to the exact calculated size of the ticket region
corrected = cv2.warpPerspective(clean_image, matrix, (maxWidth, maxHeight))
# 15. Save corrected image first, *then* open it if desired
corrected_path = os.path.join(
    os.getcwd(), 'images', 'perspective_corrected.png'
)
os.makedirs(os.path.dirname(corrected_path), exist_ok=True)
cv2.imwrite(corrected_path, corrected)

# Optional: view final corrected image
# os.system(f'mspaint "{corrected_path}"')

# 16. OCR the corrected image
gray = cv2.cvtColor(corrected, cv2.COLOR_BGR2GRAY)

config = (
    r'--oem 3 --psm 6 '
    r'-c tessedit_char_whitelist=0123456789 '
    r'-c preserve_interword_spaces=1'
)

ocr_text = pytesseract.image_to_string(
    gray, config=config
)

print("--- OCR Result ---")
for line in ocr_text.splitlines():
    print(' '.join(line[i:i+2] for i in range(0, len(line), 2)))

# convert OCR text to list of numbers
ticket_numbers = []
for line in ocr_text.splitlines():
    numbers = [int(line[i:i+2]) for i in range(0, len(line), 2) if line[i:i+2].isdigit()]
    ticket_numbers.append(numbers)

##   3. Connect to Lotto Max Website and pull the latest draw


In [ ]:

#Old method to get winning numbers from user input
#winning_numbers = input("Enter the winning numbers (space-separated): ")
#winning_numbers = [int(x) for x in winning_numbers.split()]

# input your api key from https://apiverve.com/lottery-api
api_key = input("Enter your API key from https://apiverve.com/lottery-api: ")

url = 'https://api.apiverve.com/v1/lottery'
headers = {
    'x-api-key': api_key  # Paste your copied key here
}
params = {'numbers': 'lottomax'}

response = requests.get(url, headers=headers, params=params)

lotomax_data = response.json()

winning_numbers = lotomax_data.get('data', {}).get('numbers', [])
bonus_number = lotomax_data.get('data', {}).get('bonusNumber', None)

# Convert winning numbers to a set once for efficient intersection checking
winning_set = set(winning_numbers)

# Keep track of results for each line
ticket_results = []

for idx, line in enumerate(ticket_numbers, start=1):
  line_set = set(line)

  # Calculate matches
  main_matches = len(line_set.intersection(winning_set))
  has_bonus = bonus_number in line_set

  # Determine prize tier for this specific line
  if main_matches == 7:
    prize = "JACKPOT!"
  elif main_matches == 6 and has_bonus:
    prize = "6/7 + Bonus (Major Share)"
  elif main_matches == 6:
    prize = "6/7 Share"
  elif main_matches == 5 and has_bonus:
    prize = "5/7 + Bonus Share"
  elif main_matches == 5:
    prize = "5/7 Share"
  elif main_matches == 4 and has_bonus:
    prize = "4/7 + Bonus Share"
  elif main_matches == 4 or (main_matches == 3 and has_bonus):
    prize = "$20.00 Cash"
  elif main_matches == 3:
    prize = "Free Play"
  else:
    prize = "No Prize"

  ticket_results.append({
      "line": idx,
      "numbers": line,
      "matches": main_matches,
      "bonus": has_bonus,
      "prize": prize,
  })

# Print out a clean summary of every line
print("--- Lotto Max Ticket Results ---")
for res in ticket_results:
  print(
      f"Line {res['line']} {res['numbers']} -> Main Matches:"
      f" {res['matches']} | Bonus Match: {res['bonus']} | Result:"
      f" {res['prize']}"
  )

### Export ticket_results.csv

In [ ]:
# convert ticket_results to a DataFrame for better visualization
df_results = pd.DataFrame(ticket_results)

df_results.to_csv("ticket_results.csv", index=False)

In [ ]:
# print winning #s
print("Winning numbers:", winning_numbers)

## 4. Check Max-Millions & Max-Plus

### Manually paste the max millions

In [ ]:
manual_maxmillions = """


    
    """


maxmillions_drawn = [x.strip().split() for x in manual_maxmillions.splitlines() if x.strip()]


# Check every line against every Maxmillions draw

winning_maxmillions_set = [set(draw) for draw in maxmillions_drawn]

matched_lines = []

for line_num, ticket_line in enumerate(ticket_numbers, start=1):
    for draw_num, winning_set in enumerate(winning_maxmillions_set, start=1):
        if set(ticket_line) == winning_set:
            print(f"🎉 MATCH! Line #{line_num} matched Maxmillions Draw #{draw_num}!")
            matched_lines.append(line_num)

if not matched_lines:
    print("No matches found against Maxmillions draws.")

### Manually paste the max pllus

In [ ]:
max_plus_str = """

"""

maxplus = [x.strip().split() for x in max_plus_str.splitlines() if x.strip()]

# Check every line against every Maxmillions draw

winning_maxplus_set = [set(draw) for draw in maxplus]

matched_plus_lines = []

for line_num, ticket_line in enumerate(ticket_numbers, start=1):
    for draw_num, winning_set in enumerate(winning_maxplus_set, start=1):
        if set(ticket_line) == winning_set:
            print(f"🎉 MATCH! Line #{line_num} matched Maxplus Draw #{draw_num}!")
            matched_plus_lines.append(line_num)

if not matched_plus_lines:
    print("No matches found against Maxplus draws.")